In [ ]:
# =============================================================================
# MycoIris Colony Segmentation (Step 1)
# =============================================================================
#
# Detects the colony's outer boundary on a resin-disc image and produces
# a filled binary mask and a background-removed colony image.
#
# Outputs (saved alongside the source image):
#   <base>_colony_mask.png   — filled binary mask
#   <base>_colony_clean.png  — background-removed colony
#
# Update IMG_PATH below to point to your input image.
# =============================================================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import color, filters

# ----- USER INPUTS -----------------------------------------------------------
IMG_PATH = "./data/sample.png"  # Path to colony image

# Gaussian blur sigma for L* and a* channels before thresholding.
# Lower values (1-2) preserve fine detail; higher (3-5) smooth noise.
GAUSS_SIGMA = 5

# Weight for greenness (-a* channel) in the hybrid detection score.
# Higher values emphasize green/xylindein pigment.
W_GREEN = 0.8

# Weight for darkness (-L* channel) in the hybrid detection score.
# Higher values emphasize dark, low-luminance regions.
W_DARK = 0.5

# Fraction of detected disc radius to include (prevents edge artifacts).
DISC_SHRINK = 0.99
# -----------------------------------------------------------------------------


def zscore(x, eps=1e-9):
    return (x - np.nanmean(x)) / (np.nanstd(x) + eps)

def radial_background_correct(image, cx, cy, num_bins=512, mask=None):
    """Subtracts radial average of image intensity."""
    h, w = image.shape
    yy, xx = np.indices((h, w))
    rr = np.sqrt((xx - cx)**2 + (yy - cy)**2)
    rmax = rr.max()
    rnorm = rr / (rmax + 1e-9)
    bins = np.linspace(0, 1.0, num_bins + 1)
    idx = np.digitize(rnorm, bins) - 1
    idx = np.clip(idx, 0, num_bins - 1)
    valid = np.ones_like(image, dtype=bool) if mask is None else mask.astype(bool)
    bg = np.zeros(num_bins, dtype=np.float32)
    for i in range(num_bins):
        sel = (idx == i) & valid
        bg[i] = np.mean(image[sel]) if np.any(sel) else 0
    return image - bg[idx], bg


# --- Load image and detect circular disc ---
bgr = cv2.imread(IMG_PATH, cv2.IMREAD_COLOR)
assert bgr is not None, f"Could not read image: {IMG_PATH}"
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
h, w, _ = rgb.shape

gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
gray_blur = cv2.GaussianBlur(gray, (0, 0), 3)
circles = cv2.HoughCircles(
    gray_blur, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min(h, w)//2,
    param1=100, param2=50, minRadius=min(h, w)//4, maxRadius=min(h, w)//2
)
if circles is not None:
    c = np.uint16(np.around(circles[0, 0]))
    cx, cy, r = int(c[0]), int(c[1]), int(c[2])
else:
    _, th = cv2.threshold(gray_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.bitwise_not(th)
    th = cv2.medianBlur(th, 11)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        (cx, cy), r = cv2.minEnclosingCircle(max(contours, key=cv2.contourArea))
        cx, cy, r = int(cx), int(cy), int(r)
    else:
        cx, cy, r = w//2, h//2, int(0.48 * min(h, w))

Y, X = np.indices((h, w))
RR = np.sqrt((X - cx)**2 + (Y - cy)**2)
disc_mask = RR <= int(DISC_SHRINK * r)

# --- Color conversion and cue extraction ---
rgb_f = rgb.astype(np.float32) / 255.0
lab = color.rgb2lab(rgb_f)
L = lab[:, :, 0].astype(np.float32)
a = lab[:, :, 1].astype(np.float32)

# CLAHE contrast enhancement on L* (segmentation only)
L8 = np.clip((L / 100.0) * 255.0, 0, 255).astype(np.uint8)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
L8_eq = clahe.apply(L8)
L_for_seg = (L8_eq.astype(np.float32) / 255.0) * 100.0

# Radial background correction
L_corr, _ = radial_background_correct(L, cx, cy, num_bins=512, mask=disc_mask)
L_corr_blur = cv2.GaussianBlur(L_corr, (0, 0), GAUSS_SIGMA)
a_blur = cv2.GaussianBlur(a, (0, 0), GAUSS_SIGMA)

# Hybrid score (outer ring detection)
z_dark_full = np.zeros_like(L_corr_blur)
z_green_full = np.zeros_like(a_blur)
z_dark_full[disc_mask] = zscore(-L_corr_blur[disc_mask])
z_green_full[disc_mask] = zscore(-a_blur[disc_mask])
score = W_DARK * z_dark_full + W_GREEN * z_green_full

vals = score[disc_mask]
thr = filters.threshold_otsu(vals)
mask = (score > thr) & disc_mask
mask = mask.astype(np.uint8)

# --- Find outer contour and fill interior ---
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if len(contours) == 0:
    raise RuntimeError("No contour detected; check threshold or image contrast.")

largest_contour = max(contours, key=cv2.contourArea)
filled_mask = np.zeros_like(mask)
cv2.drawContours(filled_mask, [largest_contour], -1, 1, thickness=-1)

# --- Apply mask ---
clean = rgb.copy()
clean[filled_mask == 0] = 0

# --- Save outputs ---
base = os.path.splitext(os.path.basename(IMG_PATH))[0]
out_dir = os.path.dirname(IMG_PATH)
mask_path  = os.path.join(out_dir, f"{base}_colony_mask.png")
clean_path = os.path.join(out_dir, f"{base}_colony_clean.png")

cv2.imwrite(mask_path, filled_mask * 255)
cv2.imwrite(clean_path, cv2.cvtColor(clean, cv2.COLOR_RGB2BGR))

print("Saved outputs:")
print(f"  {mask_path}")
print(f"  {clean_path}")

# --- Visualization ---
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(rgb); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(score, cmap="magma"); ax[1].set_title("Hybrid score (for outer ring)"); ax[1].axis("off")
ax[2].imshow(clean); ax[2].set_title("Background-removed colony"); ax[2].axis("off")
plt.show()


In [ ]:
# =============================================================================
# Core Removal (Step 2)
# =============================================================================
#
# Detects and removes the bright center (core) from the filled colony mask
# produced in Step 1.
#
# Outputs:
#   <base>_colony_mask_nocore.png
#   <base>_colony_clean_nocore.png
# =============================================================================

from skimage import morphology, measure

# --- Configuration ---
CORE_MAX_FRAC = 0.90    # Max fraction of colony radius to search for a core
CORE_MIN_AREA = 2000    # Ignore bright regions smaller than this (px)
CORE_SMOOTH   = 20      # Morphology disk radius for edge smoothing
CORE_METHOD   = "otsu"  # "otsu" or "percentile"
CORE_PCTL     = 85      # Percentile threshold (used when method="percentile")

# Estimate colony radius from mask area
filled_area = np.count_nonzero(filled_mask)
equiv_r = np.sqrt(filled_area / np.pi)

# Build inner search region
Y, X = np.indices(filled_mask.shape)
dist = np.hypot(X - cx, Y - cy)
inner_region = dist <= (CORE_MAX_FRAC * equiv_r)
search_region = inner_region & (filled_mask.astype(bool))

# Detect bright core using L* channel
L_vals = L[search_region]
if L_vals.size == 0:
    core_mask = np.zeros_like(filled_mask, dtype=bool)
else:
    if CORE_METHOD.lower() == "percentile":
        thr_core = np.percentile(L_vals, CORE_PCTL)
    else:
        from skimage.filters import threshold_otsu
        thr_core = threshold_otsu(L_vals)
    core_mask = (L >= thr_core) & search_region

# Retain the central connected component closest to (cx, cy)
core_mask = morphology.remove_small_objects(core_mask, min_size=CORE_MIN_AREA)
labels = measure.label(core_mask)
props = measure.regionprops(labels)
if props:
    dists = [np.hypot(p.centroid[1]-cx, p.centroid[0]-cy) for p in props]
    keep_label = props[int(np.argmin(dists))].label
    core_mask = (labels == keep_label)
else:
    core_mask = np.zeros_like(core_mask)

# Morphological smoothing
if CORE_SMOOTH and CORE_SMOOTH > 0:
    core_mask = morphology.binary_closing(core_mask, morphology.disk(CORE_SMOOTH))
    core_mask = morphology.binary_opening(core_mask, morphology.disk(max(1, CORE_SMOOTH//2)))

# Subtract core from colony mask
mask_no_core = (filled_mask.astype(bool) & ~core_mask).astype(np.uint8)

clean_no_core = rgb.copy()
clean_no_core[mask_no_core == 0] = 0

# Save outputs
base = os.path.splitext(os.path.basename(IMG_PATH))[0]
out_dir = os.path.dirname(IMG_PATH)
mask_nc_path  = os.path.join(out_dir, f"{base}_colony_mask_nocore.png")
clean_nc_path = os.path.join(out_dir, f"{base}_colony_clean_nocore.png")

cv2.imwrite(mask_nc_path,  (mask_no_core * 255).astype(np.uint8))
cv2.imwrite(clean_nc_path, cv2.cvtColor(clean_no_core, cv2.COLOR_RGB2BGR))

print("Saved core-excluded outputs:")
print(f"  {mask_nc_path}")
print(f"  {clean_nc_path}")

# Visualization
fig, ax = plt.subplots(1, 3, figsize=(16,5))
ax[0].imshow(filled_mask, cmap="gray");   ax[0].set_title("Filled colony mask"); ax[0].axis("off")
ax[1].imshow(core_mask, cmap="gray");     ax[1].set_title("Detected bright core"); ax[1].axis("off")
ax[2].imshow(clean_no_core);              ax[2].set_title("Colony (core removed)"); ax[2].axis("off")
plt.show()


In [ ]:
# =============================================================================
# Polar Unwrap and Normalization (Step 3)
# =============================================================================
#
# Converts the segmented colony from Cartesian to polar coordinates,
# producing an unwrapped rectangular representation where rows correspond
# to radial position and columns to angular position.
#
# Inputs  : <base>_colony_mask_nocore.png / <base>_colony_clean_nocore.png
#           (falls back to <base>_colony_mask.png / <base>_colony_clean.png)
# Outputs : <base>_colony_unwrapped_rgb_R{R}_A{A}.png
#           <base>_colony_unwrapped_mask_R{R}_A{A}.png
# =============================================================================

from skimage.measure import profile_line

# --- Configuration ---
ANGULAR_BINS = 512      # Columns (angular resolution)
RADIAL_BINS  = 128      # Rows (radial resolution)
INNER_FRAC   = 0.00     # 0.00 = include center
L_SMOOTH_WIN = 7        # Odd int; median window for smoothing L(theta). Set 1 to disable.

# Resolve file paths
base_dir = os.path.dirname(IMG_PATH)
base = os.path.splitext(os.path.basename(IMG_PATH))[0]

mask_candidates = [
    os.path.join(base_dir, f"{base}_colony_mask_nocore.png"),
    os.path.join(base_dir, f"{base}_colony_mask.png"),
    os.path.join(base_dir, "colony_mask_nocore.png"),
    os.path.join(base_dir, "colony_mask.png"),
]
clean_candidates = [
    os.path.join(base_dir, f"{base}_colony_clean_nocore.png"),
    os.path.join(base_dir, f"{base}_colony_clean.png"),
    os.path.join(base_dir, "colony_clean_nocore.png"),
    os.path.join(base_dir, "colony_clean.png"),
]

mask_path  = next((p for p in mask_candidates  if os.path.exists(p)), None)
clean_path = next((p for p in clean_candidates if os.path.exists(p)), None)
assert mask_path  is not None, f"Mask not found in {base_dir}."
assert clean_path is not None, f"Clean image not found in {base_dir}."

# Load inputs
mask_u8   = cv2.imread(mask_path,  cv2.IMREAD_GRAYSCALE)
clean_bgr = cv2.imread(clean_path, cv2.IMREAD_COLOR)
assert mask_u8 is not None and clean_bgr is not None, "Failed to load mask/clean images."

mask = (mask_u8 > 0).astype(np.uint8)
rgb  = cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2RGB)
H, W = mask.shape

# Colony center (centroid of mask)
M = cv2.moments(mask, binaryImage=True)
if M["m00"] > 0:
    cx = M["m10"] / M["m00"]
    cy = M["m01"] / M["m00"]
else:
    cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
cx, cy = float(cx), float(cy)

# --- Compute per-angle boundary length L(theta) ---
thetas = np.linspace(0.0, 2.0*np.pi, ANGULAR_BINS, endpoint=False)

def max_t_inside_image(cx, cy, theta, W, H):
    """Max forward distance before leaving image bounds along a ray."""
    c, s = np.cos(theta), np.sin(theta)
    ts = []
    if abs(c) > 1e-9:
        ts.append((W - 1 - cx) / c if c > 0 else (0 - cx) / c)
    if abs(s) > 1e-9:
        ts.append((H - 1 - cy) / s if s > 0 else (0 - cy) / s)
    ts = [t for t in ts if t > 0]
    return min(ts) if ts else 0.0

Ltheta = np.zeros(ANGULAR_BINS, dtype=np.float32)
for k, th in enumerate(thetas):
    tmax = max_t_inside_image(cx, cy, th, W, H)
    if tmax <= 1.0:
        Ltheta[k] = 0.0
        continue
    x2 = cx + tmax * np.cos(th)
    y2 = cy + tmax * np.sin(th)
    prof = profile_line(mask, (cy, cx), (y2, x2), order=0, mode="constant", cval=0)
    nz = np.where(prof > 0)[0]
    Ltheta[k] = float(nz[-1]) if len(nz) > 0 else 0.0

# Replace degenerate angles with local median
good = Ltheta > 1.0
if not np.any(good):
    raise RuntimeError("Could not determine any valid boundary lengths.")
L_med = np.median(Ltheta[good])
Ltheta[~good] = L_med

# Circular median smoothing
if L_SMOOTH_WIN and L_SMOOTH_WIN > 1 and L_SMOOTH_WIN % 2 == 1:
    pad = L_SMOOTH_WIN // 2
    Lpad = np.r_[Ltheta[-pad:], Ltheta, Ltheta[:pad]]
    Lsmooth = np.copy(Ltheta)
    for i in range(ANGULAR_BINS):
        Lsmooth[i] = np.median(Lpad[i:i+L_SMOOTH_WIN])
    Ltheta = Lsmooth

plt.figure(figsize=(8,3))
plt.plot(Ltheta)
plt.title("Boundary radius per angle")
plt.xlabel("Angle index")
plt.ylabel("Radius (pixels)")
plt.tight_layout()
plt.show()

# --- Build polar sampling grid and unwrap ---
r_frac = np.linspace(INNER_FRAC, 1.0, RADIAL_BINS, endpoint=True).astype(np.float32).reshape(-1, 1)
L = Ltheta.reshape(1, -1)
cos_t = np.cos(thetas, dtype=np.float64).reshape(1, -1)
sin_t = np.sin(thetas, dtype=np.float64).reshape(1, -1)
r_pix = r_frac * L
map_x = (cx + r_pix * cos_t).astype(np.float32)
map_y = (cy + r_pix * sin_t).astype(np.float32)

unwrap_rgb  = cv2.remap(rgb,        map_x, map_y, interpolation=cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_CONSTANT, borderValue=0)
unwrap_mask = cv2.remap(mask * 255, map_x, map_y, interpolation=cv2.INTER_NEAREST,
                        borderMode=cv2.BORDER_CONSTANT, borderValue=0)

# Save outputs
unwrap_rgb_path  = os.path.join(base_dir, f"{base}_colony_unwrapped_rgb_R{RADIAL_BINS}_A{ANGULAR_BINS}.png")
unwrap_mask_path = os.path.join(base_dir, f"{base}_colony_unwrapped_mask_R{RADIAL_BINS}_A{ANGULAR_BINS}.png")
cv2.imwrite(unwrap_rgb_path,  cv2.cvtColor(unwrap_rgb, cv2.COLOR_RGB2BGR))
cv2.imwrite(unwrap_mask_path, unwrap_mask)

print("Polar unwrap saved:")
print(f"  {unwrap_rgb_path}")
print(f"  {unwrap_mask_path}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(unwrap_rgb);  ax[0].set_title(f"Unwrapped RGB (R={RADIAL_BINS}, A={ANGULAR_BINS})"); ax[0].axis("off")
ax[1].imshow(unwrap_mask, cmap="gray"); ax[1].set_title("Unwrapped Mask"); ax[1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Angular Canonicalization + Mycocode v0.1 (Step 4)
# =============================================================================
#
# Determines a canonical angular orientation by fusing multi-channel
# profiles, then circularly shifts the unwrapped images to align.
# Produces a v0.1 mycocode vector (concatenated z-scored radial profiles).
#
# Inputs  : <base>_colony_unwrapped_rgb_*.png, <base>_colony_unwrapped_mask_*.png
# Outputs : <base>_colony_unwrapped_rgb_aligned.png
#           <base>_colony_unwrapped_mask_aligned.png
#           <base>_mycocode_v01.npy
#           <base>_mycocode_profiles_v01.csv
# =============================================================================

import re
import pandas as pd

# --- Configuration ---
RADIAL_BAND = (0.30, 0.80)   # Radial extent used for orientation detection
ALIGN_BY = "max"              # "max" = align to peak; "min" = align to valley
FEATURES = ["G", "a*", "b*", "Sat", "L*"]

# Multi-channel alignment
ORIENT_CHANNELS = ["G", "a*", "L*"]
WEIGHTS         = {"G": 1.0, "a*": 0.8, "L*": 0.6}

base_dir = os.path.dirname(IMG_PATH)
base = os.path.splitext(os.path.basename(IMG_PATH))[0]

def find_unwrapped(prefix_candidates):
    """Locate the most recent unwrapped file matching any of the given prefixes."""
    cands = []
    for pref in prefix_candidates:
        cands.extend([f for f in os.listdir(base_dir) if f.startswith(pref) and f.endswith(".png")])
        if cands:
            break
    if not cands:
        return None
    cands = sorted(cands, key=lambda f: os.path.getmtime(os.path.join(base_dir, f)), reverse=True)
    return os.path.join(base_dir, cands[0])

rgb_path  = find_unwrapped([f"{base}_colony_unwrapped_rgb",  "colony_unwrapped_rgb"])
mask_path = find_unwrapped([f"{base}_colony_unwrapped_mask","colony_unwrapped_mask"])
assert rgb_path and mask_path, "Unwrapped RGB/mask not found. Run the polar unwrap cell first."

# Load unwrapped images
rgb_u = cv2.cvtColor(cv2.imread(rgb_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
mask_u = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
assert rgb_u is not None and mask_u is not None, "Failed to load unwrapped images."
R, A, _ = rgb_u.shape

# Build analysis arrays
rgb_f = rgb_u.astype(np.float32) / 255.0
lab = color.rgb2lab(rgb_f)
L   = lab[:,:,0].astype(np.float32)
a   = lab[:,:,1].astype(np.float32)
b   = lab[:,:,2].astype(np.float32)
G   = rgb_f[:,:,1]
Hsv = cv2.cvtColor((rgb_f*255).astype(np.uint8), cv2.COLOR_RGB2HSV)
Sat = Hsv[:,:,1].astype(np.float32)/255.0

# --- Angular orientation via multi-channel fusion ---
r0 = int(np.clip(RADIAL_BAND[0]*R, 0, R-1))
r1 = int(np.clip(RADIAL_BAND[1]*R, 0, R))
band = slice(r0, r1)
band_mask = (mask_u[band, :] > 0)

def mean_per_angle(arr2d):
    """Compute mask-weighted mean per angular column."""
    num = (arr2d[band, :] * band_mask).sum(axis=0)
    den = np.maximum(band_mask.sum(axis=0), 1e-6)
    return num / den

profiles_ang = {}
profiles_ang["G"]   = mean_per_angle(G)
profiles_ang["a*"]  = mean_per_angle(a)
profiles_ang["L*"]  = mean_per_angle(L)
profiles_ang["b*"]  = mean_per_angle(b)
profiles_ang["Sat"] = mean_per_angle(Sat)

def zscore1d(x, eps=1e-9):
    return (x - np.nanmean(x)) / (np.nanstd(x) + eps)

# Fused z-scored profile
fused = np.zeros(A, dtype=np.float32)
for ch in ORIENT_CHANNELS:
    z = zscore1d(profiles_ang[ch])
    fused += WEIGHTS.get(ch, 1.0) * z

k_align = int(np.argmin(fused)) if ALIGN_BY.lower() == "min" else int(np.argmax(fused))

align_strength = (np.max(fused) - np.median(fused)) / (np.std(fused) + 1e-6)
print(f"Alignment: k_align={k_align} | strength={align_strength:.2f} | channels={ORIENT_CHANNELS}")

# --- Circularly shift to canonical orientation ---
def circshift_cols(img, k):
    return np.roll(img, shift=-k, axis=1)

rgb_aligned  = circshift_cols(rgb_u,  k_align)
mask_aligned = circshift_cols(mask_u, k_align)
G_aligned    = circshift_cols(G,      k_align)
a_aligned    = circshift_cols(a,      k_align)
b_aligned    = circshift_cols(b,      k_align)
Sat_aligned  = circshift_cols(Sat,    k_align)
L_aligned    = circshift_cols(L,      k_align)

# Save aligned images
aligned_rgb_path  = os.path.join(base_dir, f"{base}_colony_unwrapped_rgb_aligned.png")
aligned_mask_path = os.path.join(base_dir, f"{base}_colony_unwrapped_mask_aligned.png")
cv2.imwrite(aligned_rgb_path,  cv2.cvtColor((rgb_aligned*255).astype(np.uint8), cv2.COLOR_RGB2BGR))
cv2.imwrite(aligned_mask_path, mask_aligned)
print("Saved aligned images:")
print(f"  {aligned_rgb_path}")
print(f"  {aligned_mask_path}")

# --- Build mycocode v0.1 (radial profiles) ---
def masked_radial_profile(arr2d, mask2d):
    """Compute mask-aware mean across angular axis for each radial row."""
    num = (arr2d * (mask2d>0)).sum(axis=1)
    den = np.maximum((mask2d>0).sum(axis=1), 1e-6)
    return (num / den).astype(np.float32)

profiles = {
    "G":   masked_radial_profile(G_aligned,   mask_aligned),
    "a*":  masked_radial_profile(a_aligned,   mask_aligned),
    "b*":  masked_radial_profile(b_aligned,   mask_aligned),
    "Sat": masked_radial_profile(Sat_aligned, mask_aligned),
    "L*":  masked_radial_profile(L_aligned,   mask_aligned),
}

def zscore(x, eps=1e-9):
    x = x.astype(np.float32)
    return (x - np.nanmean(x)) / (np.nanstd(x) + eps)

code_vec = np.concatenate([zscore(profiles[k]) for k in FEATURES]).astype(np.float32)

code_path = os.path.join(base_dir, f"{base}_mycocode_v01.npy")
np.save(code_path, code_vec)

df = pd.DataFrame({
    "feature": np.repeat(FEATURES, R),
    "radial_index": np.tile(np.arange(R), len(FEATURES)),
    "value": np.concatenate([profiles[k] for k in FEATURES])
})
csv_path = os.path.join(base_dir, f"{base}_mycocode_profiles_v01.csv")
df.to_csv(csv_path, index=False)

print(f"Mycocode v0.1 saved (length {code_vec.size}):")
print(f"  {code_path}")
print(f"  {csv_path}")

# Visualization
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax = ax.ravel()
ax[0].imshow(rgb_u); ax[0].set_title("Unwrapped (input)"); ax[0].axis("off")

ax[1].plot(zscore1d(profiles_ang["G"]),  label="G (z)")
ax[1].plot(zscore1d(profiles_ang["a*"]), label="a* (z)")
ax[1].plot(zscore1d(profiles_ang["L*"]), label="L* (z)")
ax[1].plot(fused, label="Fused (z-weighted)", linewidth=2)
ax[1].axvline(k_align, color="k", linestyle="--", label="k_align")
ax[1].set_title("Angular profile (multi-channel, fused)")
ax[1].legend(loc="upper right", fontsize=8)

ax[2].imshow(rgb_aligned); ax[2].set_title("Aligned unwrapped"); ax[2].axis("off")
ax[3].plot(profiles["G"]); ax[3].set_title("Radial profile: G (aligned)")
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Iris-Style Binary Mycocode Encoding (Step 5)
# =============================================================================
#
# Applies a multi-channel, multi-scale Gabor filter bank to the aligned
# unwrapped colony image and encodes the complex filter responses into
# 2-bit phase codes with mask-aware validity.
#
# Outputs: <base>_mycocode_bits.npz
# =============================================================================

from math import pi
from scipy.ndimage import gaussian_filter

# --- Configuration ---
CHANNELS  = ["G", "a*", "L*"]
SCALES    = [3, 6, 9]               # Gabor sigma values
ORIENTS   = [0, pi/4, pi/2, 3*pi/4] # Gabor orientations
KERNEL_SZ = 21
ENERGY_Q  = 0.05    # Drop lowest 5% energy among valid pixels
COVER_MIN = 0.30    # Require >= 30% valid-mask coverage under kernel

base_dir = os.path.dirname(IMG_PATH)
base = os.path.splitext(os.path.basename(IMG_PATH))[0]

def find_latest(prefix_candidates):
    """Find the most recent PNG matching any candidate prefix."""
    for pref in prefix_candidates:
        cands = [f for f in os.listdir(base_dir) if f.startswith(pref) and f.endswith(".png")]
        if cands:
            cands = sorted(cands, key=lambda f: os.path.getmtime(os.path.join(base_dir, f)), reverse=True)
            return os.path.join(base_dir, cands[0])
    return None

rgb_path  = find_latest([f"{base}_colony_unwrapped_rgb_aligned",
                         f"{base}_colony_unwrapped_rgb",
                         "colony_unwrapped_rgb_aligned",
                         "colony_unwrapped_rgb"])
mask_path = find_latest([f"{base}_colony_unwrapped_mask_aligned",
                         f"{base}_colony_unwrapped_mask",
                         "colony_unwrapped_mask_aligned",
                         "colony_unwrapped_mask"])
assert rgb_path and mask_path, "Run unwrap first."

# Load
rgb_u  = cv2.cvtColor(cv2.imread(rgb_path), cv2.COLOR_BGR2RGB)
mask_u = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
R, A, _ = rgb_u.shape

# Hard and soft masks
mask_bool = (mask_u > 0)
mask_f    = mask_bool.astype(np.float32)
soft_radius = max(1, KERNEL_SZ // 4)
mask_soft = gaussian_filter(mask_f, soft_radius)
mask_soft = np.clip(mask_soft, 0.0, 1.0)

# Compute Lab color space
rgb_f = rgb_u.astype(np.float32) / 255.0
lab_f = color.rgb2lab(rgb_f)

def channel_src(name: str) -> np.ndarray:
    """Extract and normalize a single channel for Gabor filtering."""
    if name == "G":
        src = rgb_f[:, :, 1]
    elif name == "a*":
        src = lab_f[:, :, 1].astype(np.float32)
    elif name == "L*":
        src = (lab_f[:, :, 0].astype(np.float32) / 100.0)
    else:
        raise ValueError(f"Unsupported channel: {name}")
    # Row-wise illumination normalization, then feather by soft mask
    row_mean = np.divide((src * mask_f).sum(axis=1), mask_f.sum(axis=1) + 1e-6)[:, None]
    return (src - row_mean) * mask_soft


def complex_gabor(ksize, sigma, theta, lam=None, gamma=0.5, psi=0.0):
    """Generate a complex Gabor kernel (real and imaginary parts)."""
    half = ksize // 2
    y, x = np.mgrid[-half:half+1, -half:half+1].astype(np.float32)
    xr =  x*np.cos(theta) + y*np.sin(theta)
    yr = -x*np.sin(theta) + y*np.cos(theta)
    g  = np.exp(-(xr**2 + (gamma**2)*(yr**2)) / (2*sigma**2))
    lam = lam if lam is not None else (2*np.pi*sigma/1.5)
    phase = (2*np.pi*xr/lam) + psi
    re = g * np.cos(phase); im = g * np.sin(phase)
    for k in (re, im):
        k -= k.mean(); n = np.linalg.norm(k) + 1e-6; k /= n
    return re.astype(np.float32), im.astype(np.float32)

# Build filter bank
bank = []
for s in SCALES:
    for th in ORIENTS:
        kre, kim = complex_gabor(KERNEL_SZ, sigma=s, theta=th, lam=None, gamma=0.5, psi=0.0)
        bank.append((kre, kim, s, th))
Nf = len(bank)
Nc = len(CHANNELS)

abs_k = [np.abs(kre) for (kre, _, _, _) in bank]

# --- Mask-aware convolution and 2-bit quantization ---
bits      = np.zeros((Nc, Nf, R, A, 2), dtype=np.uint8)
mask_bits = np.zeros((Nc, Nf, R, A),     dtype=np.uint8)
valid_ratios = []

# Precompute per-filter coverage
coverage_list = []
for fi in range(Nf):
    cov = cv2.filter2D(mask_soft, cv2.CV_32F, abs_k[fi], borderType=cv2.BORDER_REPLICATE)
    coverage_list.append(cov / (abs_k[fi].sum() + 1e-6))

for ci, ch in enumerate(CHANNELS):
    src = channel_src(ch).astype(np.float32)
    for fi, (kre, kim, s, th) in enumerate(bank):
        coverage = coverage_list[fi]

        re_raw = cv2.filter2D(src, cv2.CV_32F, kre, borderType=cv2.BORDER_REPLICATE)
        im_raw = cv2.filter2D(src, cv2.CV_32F, kim, borderType=cv2.BORDER_REPLICATE)

        re_resp = re_raw / (coverage + 1e-6)
        im_resp = im_raw / (coverage + 1e-6)
        mag = np.hypot(re_resp, im_resp)

        base_valid = (mask_bool & (coverage >= COVER_MIN))
        thr = np.quantile(mag[base_valid], ENERGY_Q) if np.any(base_valid) else 0.0
        valid = (base_valid & (mag >= thr))

        # 2-bit phase quantization
        bits[ci, fi, :, :, 0] = (re_resp >= 0).astype(np.uint8)
        bits[ci, fi, :, :, 1] = (im_resp >= 0).astype(np.uint8)

        inv = ~valid
        bits[ci, fi, inv, 0] = 0
        bits[ci, fi, inv, 1] = 0

        mask_bits[ci, fi] = valid.astype(np.uint8)
        valid_ratios.append(valid.sum() / valid.size)

# Reshape to (R, A, Nc*Nf)
bits_RA = np.transpose(bits, (2, 3, 0, 1, 4)).reshape(R, A, Nc*Nf, 2)
mask_RA = np.transpose(mask_bits, (2, 3, 0, 1)).reshape(R, A, Nc*Nf)

# Pack 2 bits into values 0..3
code_uint2 = (bits_RA[..., 0].astype(np.uint8) << 1) | bits_RA[..., 1].astype(np.uint8)

# Metadata
channels_per_filter = np.array([CHANNELS[ci] for ci in range(Nc) for _ in range(Nf)], dtype=object)
filters_meta = np.array([(bank[fi][2], float(bank[fi][3])) for _ci in range(Nc) for fi in range(Nf)], dtype=np.float32)

# Save
out_npz = os.path.join(base_dir, f"{base}_mycocode_bits.npz")
np.savez_compressed(out_npz,
    code_uint2=code_uint2.astype(np.uint8),
    mask=mask_RA.astype(np.uint8),
    R=np.int32(R), A=np.int32(A),
    filters=filters_meta,
    channels=channels_per_filter,
    channel_set=np.array(CHANNELS, dtype=object),
    energy_q=np.array([ENERGY_Q], dtype=np.float32),
    cover_min=np.array([COVER_MIN], dtype=np.float32),
    kernel_sz=np.array([KERNEL_SZ], dtype=np.int32)
)

Nf_total = Nc * Nf
print(f"Saved iris-style mycocode: {out_npz}")
print(f"Code shape: {code_uint2.shape}  (R, A, filters={Nf_total})")
print(f"Mask shape: {mask_RA.shape}     (R, A, filters={Nf_total})")
print(f"Filters per channel: {Nf} | Channels: {CHANNELS}")
valid_ratios = np.array(valid_ratios, dtype=np.float32)
print(f"Valid pixel ratios — mean +/- std: {valid_ratios.mean():.3f} +/- {valid_ratios.std():.3f}")

# Sanity visualization
fi_view = 0
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(rgb_u);                           ax[0].set_title("Unwrapped RGB"); ax[0].axis("off")
ax[1].imshow(mask_bool, cmap="gray");          ax[1].set_title("Unwrapped Mask"); ax[1].axis("off")
ax[2].imshow(code_uint2[:, :, fi_view], cmap="tab20"); ax[2].set_title("Code bins (one filter)"); ax[2].axis("off")
ax[3].imshow(mask_RA[:, :, fi_view], cmap="gray");      ax[3].set_title("Validity (one filter)"); ax[3].axis("off")
plt.tight_layout(); plt.show()

avg_valid = mask_RA.mean(axis=2)
plt.figure(figsize=(8, 3))
plt.imshow(avg_valid, cmap="viridis", aspect="auto", vmin=0, vmax=1)
plt.title("Average validity across filters")
plt.axis("off")
plt.show()


In [ ]:
# =============================================================================
# Mycocode Visualization — Bit Mosaic and Per-Filter Views (Step 6)
# =============================================================================
#
# Loads the encoded mycocode NPZ and produces diagnostic visualizations:
#   - A barcode-like bit mosaic across all filters
#   - A per-filter 2-bit bin view with its validity mask
#
# Outputs : <base>_mycocode_mosaic.png
#           <base>_mycocode_filterview_*.png
# =============================================================================

# Resolve NPZ path
base_dir = os.path.dirname(IMG_PATH)
base = os.path.splitext(os.path.basename(IMG_PATH))[0]

npz_candidates = [
    os.path.join(base_dir, f"{base}_mycocode_bits.npz"),
    os.path.join(base_dir, "mycocode_bits.npz"),
]
npz_path = next((p for p in npz_candidates if os.path.exists(p)), None)
assert npz_path is not None, f"Missing mycocode NPZ in {base_dir}."

data = np.load(npz_path, allow_pickle=True)
code = data["code_uint2"]
mask = data["mask"]
R, A, Nf = code.shape

# Derive bitplanes from uint2 values
bit_re = ((code & 0b10) > 0).astype(np.uint8)
bit_im = ((code & 0b01) > 0).astype(np.uint8)

# Build barcode-like mosaic: [Re|Im] per filter, concatenated
tiles_bits, tiles_mask = [], []
for f in range(Nf):
    block_bits = np.concatenate([bit_re[:, :, f], bit_im[:, :, f]], axis=1)
    block_mask = np.concatenate([mask[:, :, f],   mask[:, :, f]],   axis=1)
    tiles_bits.append(block_bits)
    tiles_mask.append(block_mask)

mosaic_bits = np.concatenate(tiles_bits, axis=1)
mosaic_mask = np.concatenate(tiles_mask, axis=1)

# Invalid pixels rendered as mid-gray
vis = mosaic_bits.astype(np.float32)
vis[mosaic_mask == 0] = 0.5

plt.figure(figsize=(14, 4))
plt.imshow(vis, cmap="gray", aspect="auto")
plt.title(f"{base} — Mycocode bit mosaic (R={R}, A={A}, filters={Nf})")
plt.axis("off")
mosaic_path = os.path.join(base_dir, f"{base}_mycocode_mosaic.png")
plt.savefig(mosaic_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved mycocode mosaic: {mosaic_path}")

# --- Per-filter view ---
fi_view = 0

channels_per_filter = data["channels"]
filters_meta = data["filters"]

chan = str(channels_per_filter[fi_view])
sigma, theta = filters_meta[fi_view]
bins_img = code[:, :, fi_view]
valid_img = mask[:, :, fi_view].astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax = ax.ravel()
im0 = ax[0].imshow(bins_img, cmap="tab20", aspect="auto")
ax[0].set_title(f"{base} — {chan}, sigma={sigma:.1f}, theta={theta/np.pi:.2f}pi — 2-bit bins")
ax[0].axis("off")
ax[1].imshow(valid_img, cmap="gray", aspect="auto")
ax[1].set_title(f"Validity mask — {chan}, sigma={sigma:.1f}, theta={theta/np.pi:.2f}pi")
ax[1].axis("off")

plt.tight_layout()
flt_path = os.path.join(
    base_dir,
    f"{base}_mycocode_filterview_c{chan}_s{int(round(sigma))}_t{int(round(180*theta/np.pi))}.png"
)
plt.savefig(flt_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved per-filter view: {flt_path}")
